## 1 setup

In [41]:
%pip install -q "datasets==3.6.0" pandas pyarrow numpy tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from pathlib import Path
import json
import math
import warnings #gestire warnings
import numpy as np 
import pandas as pd
import pyarrow.parquet as pq 
from tqdm.auto import tqdm #barra di progresso

warnings.filterwarnings("ignore", category=FutureWarning) #nasconde avvisi per avere output meno rumoroso
#impostazioni di pandas
pd.set_option("display.max_columns",100)
pd.set_option("display.max_colwidth",140)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")  

c:\Users\arsid\Desktop\Tesi-ecommerce\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2 percorsi e configurazione scelta nel notebook 02

In [ ]:
CATEGORY="Electronics" #categoria scelta
SELECTED_CONFIG="user3_item8" #configurazione scelta
POSITIVE_RATING_THRESHOLDS=4 #rating minimo per considerare una interazione positiva
MIN_USER_INTERACTIONS=3 
MIN_ITEM_INTERACTIONS=8
N_NEG=100 #quanti item negativi campionare per ogni esempio di validation/test
RANDOM_STATE=42 #seed casuale per rendere il negative sampling riproducibile


In [ ]:
PROJECT_ROOT=Path(r"C:\Users\arsid\Desktop\Tesi-ecommerce")
SOURCE_DIR=PROJECT_ROOT/"data"/"processed"/"electronics_intersection" #output dei precedenti notebook
ANALYSIS_DIR=SOURCE_DIR/"analysis" #riepiloghi notebook 2
OUTPUT_DIR=PROJECT_ROOT/"data"/"processed"/"electronics_recommender" #output di questo notebook03
OUTPUT_DIR.mkdir(parents=True,exist_ok=True) #crea cartella output se non esiste

CANDIDATE_INTERACTIONS_PATH=SOURCE_DIR/f"candidate_interactions_{SELECTED_CONFIG}.parquet" #interazioni candidate
CANDIDATE_ITEMS_PATH=SOURCE_DIR/f"candidate_items_{SELECTED_CONFIG}.parquet" #prodotti candidati
CANDIDATE_QA_PATH=SOURCE_DIR/f"candidate_qa_{SELECTED_CONFIG}.parquet"  #qa candidati
NOTEBOOK02_CONFIG_PATH=ANALYSIS_DIR/"notebook02_selected_filter_config.json" #riepilogo config scelta

print("Project root:",PROJECT_ROOT)
print("Source dir:",SOURCE_DIR)
print("Output dir:",OUTPUT_DIR)

Project root: C:\Users\arsid\Desktop\Tesi-ecommerce
Source dir: C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\electronics_intersection
Output dir: C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\electronics_recommender
candidate_interactions  ->C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\electronics_intersection\candidate_interactions_user3_item8.parquet | exists=True
candidate_items         ->C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\electronics_intersection\candidate_items_user3_item8.parquet | exists=True
candidate_qa            ->C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\electronics_intersection\candidate_qa_user3_item8.parquet | exists=True
notebook02_config       ->C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\electronics_intersection\analysis\notebook02_selected_filter_config.json | exists=True


## 3 caricamento dei dati candidati

In [46]:
#carichiamo dati
interactions_df=pd.read_parquet(CANDIDATE_INTERACTIONS_PATH)
items_df=pd.read_parquet(CANDIDATE_ITEMS_PATH)
qa_df=pd.read_parquet(CANDIDATE_QA_PATH)
notebook02_Config=json.loads(NOTEBOOK02_CONFIG_PATH.read_text(encoding="utf-8"))
#stampiamo dimensioni tabelle
print("interactions_df:",interactions_df.shape)
print("items_df:",items_df.shape)
print("qa_df:",qa_df.shape)
display(pd.DataFrame([notebook02_Config])) #riepilogo dati dentro configurazione scelta

interactions_df: (844060, 7)
items_df: (15069, 12)
qa_df: (128151, 8)


,selected_config_name,min_user_interactions,min_item_interactions,num_users,num_items,num_interactions,density_pct,sparsity_pct
0,user3_item8,3,8,196509,15069,844060,0.028504,99.971496


## 4 prepariamo timestamp

In [47]:
TIME_COL="datetime" #colonna temporale usata nello split

if TIME_COL in interactions_df.columns: 
    interactions_df[TIME_COL]=pd.to_datetime( #convertiamola in datetime pandas 
        interactions_df[TIME_COL], 
        errors="coerce", #trasforma errori in NaT
    )
elif "timestamp" in interactions_df.columns: #se datetime non c'è, usa timestamp
    interactions_df[TIME_COL]=pd.to_datetime(
        interactions_df["timestamp"],
        unit="ms",
        errors="coerce",
    )
else:
    raise ValueError("mancano sia timestamp sia datetime")

invalid_datetime_rows=int(interactions_df[TIME_COL].isna().sum()) #conta righe senza datetime valido
if invalid_datetime_rows>0:
    print("righe rimosse senza datetime valido:", invalid_datetime_rows)
    interactions_df=interactions_df.dropna(subset=[TIME_COL]).copy() #rimuove righe


print("Colonna temporale usata per lo split:",TIME_COL)
print("Data minima:",interactions_df[TIME_COL].min())
print("Data massima:",interactions_df[TIME_COL].max()) 

preview_c=["timestamp",TIME_COL] if "timestamp" in interactions_df.columns else [TIME_COL]
display(interactions_df[preview_c].head())

Colonna temporale usata per lo split: datetime
Data minima: 1999-08-21 22:04:38
Data massima: 2023-06-30 05:28:46.120000


,timestamp,datetime
0,1275251437000,2010-05-30 20:30:37
1,1311718723000,2011-07-26 22:18:43
2,1344043262000,2012-08-04 01:21:02
3,1389108747000,2014-01-07 15:32:27
4,1381408252000,2013-10-10 12:30:52


## 5 definizione target: feedback positivo
consideriamo coppie come rilevanti se rating è almeno 4

In [48]:
rating_dist=( 
    #distribuzione dei rating nel dataset candidato, la rifacciamo perchè potrebbe essere
    #diverso dopo aver filtrato con la configurazione scelta
    interactions_df["rating"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("rating")
    .reset_index(name="count")
)
rating_dist["percentage"]=rating_dist["count"]/len(interactions_df)*100
display(rating_dist)

,rating,count,percentage
0,1.000000,61551,7.292254
1,2.000000,41741,4.945265
2,3.000000,68865,8.158780
3,4.000000,152080,18.017676
4,5.000000,519823,61.586025


In [49]:
positive_interactions_df=interactions_df[interactions_df["rating"]>= POSITIVE_RATING_THRESHOLDS].copy() #tiene solo rating pos
positive_interactions_df["interaction_value"]=1 #crea colonna binaria cosi capiamo i rating positivi
target_stats={
    "original_interaction":int(len(interactions_df)), #numero di interazioni candidate originali
    "positive_interaction":int(len(positive_interactions_df)), #n int con rating >=soglia
    "removed_interaction":int(len(interactions_df)-len(positive_interactions_df)), #num int escluse
    "positive_int_percentage":float(len(positive_interactions_df)/len(interactions_df)*100), # % interazioni positive
    "original_users_num":int(interactions_df["user_id"].nunique()), #utenti prima
    "positive_users_num":int(positive_interactions_df["user_id"].nunique()), #utenti con almeno un rating positivo
    "original_items":int(interactions_df["parent_asin"].nunique()), #items prima
    "positive_items":int(positive_interactions_df["parent_asin"].nunique()), #prodotti con almeno un rating positivo
}

target_stats_df=pd.DataFrame.from_dict(target_stats,orient="index",columns=["value"]) #tabella
display(target_stats_df)

,value
original_interaction,"844,060.000000"
positive_interaction,"671,903.000000"
removed_interaction,"172,157.000000"
positive_int_percentage,79.603701
original_users_num,"196,509.000000"
positive_users_num,"192,246.000000"
original_items,"15,069.000000"
positive_items,"15,069.000000"


## 6 Rifiltraggio iterativo dopo la scelta del target
Dopo aver tenuto solo rating>=4 dobbiamo ricontrollare che tutti gli utenti abbiamo >=3 interazioni e tutti i prodotti >=8 interazioni

In [50]:
def iter_filter(df, min_user_interactions,min_item_interactions,user_col="user_id",item_col="parent_asin",max_iter=50):
    #funzione per filtro iterativo
    current_df=df.copy()
    history_rows=[]
    for iteration in range(1,max_iter+1):
        #numeri prima del filtro
        rows_before=len(current_df)
        users_before=current_df[user_col].nunique()
        items_before=current_df[item_col].nunique()
        if len(current_df)==0: #se vuoto il dataframe
            break
        item_num=current_df[item_col].value_counts() #conta interazioni positive per prodotto
        valid_items=item_num[item_num>=min_item_interactions].index #seleziona prodotti sopra la soglia
        current_df=current_df[current_df[item_col].isin(valid_items)].copy() #mantiene solo prodotti validi
        user_counts=current_df[user_col].value_counts() #conta int positiv per utente
        valid_users=user_counts[user_counts>=min_user_interactions].index #seleziona utenti sopra soglia
        current_df=current_df[current_df[user_col].isin(valid_users)].copy() #tiene solo utenti validi
        #contiamo righe dopo filtro
        rows_after=len(current_df)
        users_after=current_df[user_col].nunique()
        items_after=current_df[item_col].nunique()
        history_rows.append({ #statistiche dell'iterazione
            "iteration":iteration,
            "rows_before":rows_before,
            "rows_after":rows_after,
            "rows_removed":rows_before-rows_after,
            "users_before":users_before,
            "users_after":users_after,
            "users_removed":users_before-users_after,
            "items_before":items_before,
            "items_after":items_after,
            "items_removed":items_before-items_after,
        })
        if rows_after==rows_before:
            break

    history_df=pd.DataFrame(history_rows)
    return current_df.reset_index(drop=True), history_df


In [51]:

filtered_positive_df, filter_history_df=iter_filter(
    positive_interactions_df,
    min_user_interactions=MIN_USER_INTERACTIONS,
    min_item_interactions=MIN_ITEM_INTERACTIONS,
    user_col="user_id",
    item_col="parent_asin",
    max_iter=50,
)
print("interazioni positive dopo filtraggio:", len(filtered_positive_df))
print("utenti dopo filtraggio:",filtered_positive_df["user_id"].nunique())
print("prodotti dopo filtraggio:",filtered_positive_df["parent_asin"].nunique())
display(filter_history_df)

interazioni positive dopo filtraggio: 550548
utenti dopo filtraggio: 131398
prodotti dopo filtraggio: 12306


,iteration,rows_before,rows_after,rows_removed,users_before,users_after,users_removed,items_before,items_after,items_removed
0,1,671903,563108,108795,192246,134209,58037,15069,13396,1673
1,2,563108,551692,11416,134209,131657,2552,13396,12396,1000
2,3,551692,550657,1035,131657,131421,236,12396,12315,81
3,4,550657,550548,109,131421,131398,23,12315,12306,9
4,5,550548,550548,0,131398,131398,0,12306,12306,0


In [ ]:
final_user_counts=filtered_positive_df["user_id"].value_counts()
final_item_counts=filtered_positive_df["parent_asin"].value_counts()

all_user_thresholds_ok=bool(final_user_counts.min()>=MIN_USER_INTERACTIONS) if len(final_user_counts) else False
all_item_thresholds_ok=bool(final_item_counts.min()>=MIN_ITEM_INTERACTIONS) if len(final_item_counts) else False

post_filter_check={ #dizionario per controlli
    "min_user_int":int(final_user_counts.min())if len(final_user_counts)else 0,
    "min_item_int":int(final_item_counts.min())if len(final_item_counts)else 0,
    "users_with_3int":int((final_user_counts>=MIN_USER_INTERACTIONS).sum()),
    "item_with_8int":int((final_item_counts>=MIN_ITEM_INTERACTIONS).sum()),
    "all_thresholds_ok":bool(all_user_thresholds_ok and all_item_thresholds_ok),
}

display(pd.DataFrame.from_dict(post_filter_check,orient="index",columns=["value"]))
if not post_filter_check["all_thresholds_ok"]:
    raise ValueError("Dopo filtro non tutte le soglie sono rispettate")


,value
min_user_int,3
min_item_int,8
users_with_3int,131398
item_with_8int,12306
all_thresholds_ok,True


## 7 filtraggio prodotti e qa

In [ ]:
final_products=set(filtered_positive_df["parent_asin"].unique()) #crea insieme prodotti finali
final_items_df=items_df[items_df["parent_asin"].isin(final_products)].copy() #filtra i metadati sui prodotti finali
final_qa_df=qa_df[qa_df["parent_asin"].isin(final_products)].copy() #filtra qa

print("final_items_df:",final_items_df.shape)
print("final_qa_df:",final_qa_df.shape)
print("Prodotti finali con metadati:",final_items_df["parent_asin"].nunique())
print("Prodotti finali con QA:",final_qa_df["parent_asin"].nunique())

final_items_df: (12306, 12)
final_qa_df: (106914, 8)
Prodotti finali con metadati: 12306
Prodotti finali con QA: 12306


## 8 creiamo indici numerici

In [54]:
user_mapping_df=pd.DataFrame({"user_id":sorted(filtered_positive_df["user_id"].unique())})#tab utenti ordinata
user_mapping_df["user_idx"]=np.arange(len(user_mapping_df),dtype=np.int64)

item_mapping_df=pd.DataFrame({"parent_asin":sorted(filtered_positive_df["parent_asin"].unique())})#tab prodotti ordinata
item_mapping_df["item_idx"]=np.arange(len(item_mapping_df),dtype=np.int64)

indexed_interactions_df=filtered_positive_df.merge(user_mapping_df,on="user_id",how="left")
indexed_interactions_df=indexed_interactions_df.merge(item_mapping_df,on="parent_asin",how="left")

final_items_df=final_items_df.merge(item_mapping_df,on="parent_asin",how="left")
final_qa_df=final_qa_df.merge(item_mapping_df,on="parent_asin",how="left")

print("Utenti mappati:",len(user_mapping_df))
print("Prodotti mappati:",len(item_mapping_df))
display(user_mapping_df.head())
display(item_mapping_df.head())

Utenti mappati: 131398
Prodotti mappati: 12306


,user_id,user_idx
0,AE22236AFRRSMQIKGG7TPTB75QEA,0
1,AE222H3FGXWLHRFUMGMS2RR57NDQ,1
2,AE223NW3QAU2KTAY5MAEU6VPBAUA,2
3,AE224HM2QAW5TTSDL2QRDERA6KMA,3
4,AE2252DKW4XJIZP5QPFMQVJBVRTA,4


,parent_asin,item_idx
0,0972683275,0
1,1400501466,1
2,1400532655,2
3,1400532736,3
4,1400599997,4


In [55]:
#in caso di errori dopo il merge, a causa di duplicati o NaN riportiamo a interi, teoricamente è tutto già stato controllato, ma per sicurezza lo facciamo
indexed_interactions_df["user_idx"]=indexed_interactions_df["user_idx"].astype(np.int64)
indexed_interactions_df["item_idx"]=indexed_interactions_df["item_idx"].astype(np.int64)
final_items_df["item_idx"]=final_items_df["item_idx"].astype(np.int64)
if "item_idx" in final_qa_df.columns:
    final_qa_df["item_idx"]=final_qa_df["item_idx"].astype(np.int64)


## 9 split temporale train/val/test
ultima in test, penultima in val, il resto in training

In [56]:
indexed_interactions_df["interaction_id"]=np.arange(len(indexed_interactions_df),dtype=np.int64) #crea id interazione
indexed_interactions_df=indexed_interactions_df.sort_values(
    ["user_idx",TIME_COL,"interaction_id"], #ordina
    ascending=[True,True,True],
).reset_index(drop=True) #ricrea l'indice

indexed_interactions_df["n_userhistory"]=indexed_interactions_df.groupby("user_idx").cumcount() #posizione temporale fra le varie interazioni dell'utente
indexed_interactions_df["user_history_length"]=indexed_interactions_df.groupby("user_idx")["item_idx"].transform("size") #num tot int utente
#ultima in test, penultima in val, altre in training
indexed_interactions_df["split"]="train"
validation_mask=indexed_interactions_df["n_userhistory"]==indexed_interactions_df["user_history_length"]-2
test_mask=indexed_interactions_df["n_userhistory"]==indexed_interactions_df["user_history_length"]-1

indexed_interactions_df.loc[validation_mask,"split"]="validation"
indexed_interactions_df.loc[test_mask,"split"]="test"
#estraiamo i set
train_df=indexed_interactions_df[indexed_interactions_df["split"]=="train"].copy()
validation_df=indexed_interactions_df[indexed_interactions_df["split"]=="validation"].copy()
test_df=indexed_interactions_df[indexed_interactions_df["split"]=="test"].copy()

print("Train interactions:",len(train_df))
print("validation interactions:",len(validation_df))
print("test interactions:",len(test_df))

Train interactions: 287752
validation interactions: 131398
test interactions: 131398


In [57]:
split_summary_df=(
    indexed_interactions_df.groupby("split")
    .agg(
        num_rows=("interaction_id","size"),
        num_users=("user_idx","nunique"),
        num_items=("item_idx","nunique"),
        min_time=(TIME_COL,"min"),
        max_time=(TIME_COL,"max"), 
    )
    .reset_index()
)
display(split_summary_df)

,split,num_rows,num_users,num_items,min_time,max_time
0,test,131398,131398,11507,2000-11-26 20:22:16,2023-06-23 14:41:11.200
1,train,287752,131398,12299,1999-08-21 22:04:38,2023-06-23 14:38:21.166
2,validation,131398,131398,11887,2000-10-31 17:15:59,2023-06-23 14:40:37.747


## 10 controllo cold start sugli item
non vogliamo prodotti nel val e test mai visti nel training

In [58]:
#insieme prodotti dei vari split
train_items=set(train_df["item_idx"].unique())
validation_items=set(validation_df["item_idx"].unique())
test_items=set(test_df["item_idx"].unique())

validation_cold_items=validation_items-train_items #prod val assenti dal train
test_cold_items=test_items-train_items #prod test assenti dal train

#val e test migliorati, per uso di collaborative filtering
validation_eval_df=validation_df[validation_df["item_idx"].isin(train_items)].copy()
test_eval_df=test_df[test_df["item_idx"].isin(train_items)].copy()

cold_start_stats={ #statistiche cold start
    "train_items":int(len(train_items)),
    "validation_items":int(len(validation_items)),
    "test_items":int(len(test_items)),
    "validation_cold_items":int(len(validation_cold_items)),
    "test_cold_items":int(len(test_cold_items)),
    "validation_rows_original":int(len(validation_df)),
    "validation_rows_eval":int(len(validation_eval_df)),
    "test_rows_original":int(len(test_df)),
    "test_rows_eval":int(len(test_eval_df)),
}

display(pd.DataFrame.from_dict(cold_start_stats, orient="index", columns=["value"]))

,value
train_items,12299
validation_items,11887
test_items,11507
validation_cold_items,7
test_cold_items,7
validation_rows_original,131398
validation_rows_eval,131376
test_rows_original,131398
test_rows_eval,131352


## 11 negative sampling
per confrontare l'item positivo reale, usiamo N_NEG prodotti del train escludendo prodotti positivi per quell'utente

In [59]:
user_seen_items=( #elenco item positivi per ogni utente
    indexed_interactions_df.groupby("user_idx")["item_idx"] #raggruppa int positive per utente e prende solo colonna item_idx
    .apply(lambda values: set(values.astype(int))) #converte la lista di item in insieme
    .to_dict() #converte in dizionario
)
candidate_item_idx=np.array(sorted(train_items), dtype=np.int64) #usa come candidati neg solo item presenti nel train
print("Numero item candidati per negative samples:",len(candidate_item_idx))

Numero item candidati per negative samples: 12299


In [60]:
#campionamento negativi, cioè prodotti che l'utente non ha già valutato positivamente
#useridx utente per cui stai generando i negativi
#positiveitemidx id del item pos corrente
#candidateitems insieme item da cui puoi pescare negativi
#seenitemsbyuser dizionario per ogni utente, tutti gli item positivi osservati
#n_neg=100 
#rng generatore casuale
def sample_neg_items(user_idx, positive_item_idx, candidate_items, seen_items_byuser, n_neg, rng):
    seen_items=seen_items_byuser.get(int(user_idx),set()) #recupera gli item pos già osservati dell'utente
    negative_items=[] #lista dei negativi
    negative_set=set() #insieme per evitare duplicati
    max_rounds=20
    sample_size=min(len(candidate_items),max(n_neg*3,n_neg+len(seen_items))) #quanti candidati estrarre per tentativo, max tra 300 e 100+item visti
    for _ in range(max_rounds): #ripete 20 volte
        sampled_items=rng.choice(candidate_items,size=sample_size, replace=False)#estrae candidati senza ripetizioni nel tentativo
        for sampled_item in sampled_items:
            sampled_item=int(sampled_item)
            if sampled_item ==int(positive_item_idx): #evita di campionare item positivo reale
                continue
            if sampled_item in seen_items: #evita item positivi per l'utente
                continue
            if sampled_item in negative_set: #evita duplicati tra i negativi
                continue
            negative_items.append(sampled_item) #aggiunge candidato valido ai negativi
            negative_set.add(sampled_item) #lo mette nell'insieme
            if len(negative_items)>=n_neg: #se abbiamo abbastanza negativi
                return negative_items
    available_items=[int(item) for item in candidate_items if int(item) not in seen_items and int(item) != int(positive_item_idx)]
    #prendi tutti gli item presenti in candidate_items,tieni solo quelli che l’utente non ha già visto positivamente,e togli anche l’item positivo reale della riga corrente
    available_items=[item for item in available_items if item not in negative_set] #rimuove negativi già scelti
    #Prendi la lista degli item ancora disponibili e rimuovi quelli che sono già stati scelti come negativi.

    remaining_needed=n_neg-len(negative_items) #calcola quanti prodotti negativi mancano
    if len(available_items)<remaining_needed:
        raise ValueError(f"Non ci sono abbastanza negativi per user_idx={user_idx}")
    extra_items=rng.choice(np.array(available_items, dtype=np.int64),size=remaining_needed,replace=False)
    #Estrae casualmente gli ultimi negativi mancanti dalla lista completa degli item ancora disponibili.
    negative_items.extend([int(item)for item in extra_items]) #aggiunge questi ultimi item alla lista dei negativi.
    return negative_items

In [61]:
#evaldf il dataframe da valutare
#splitname il nome dello split
def build_negative_samples(eval_df, split_name, n_neg=N_NEG, seed=RANDOM_STATE): #costruzione negative samples
    rng=np.random.default_rng(seed) #crea generatore casuale, ma riproducibile
    sample_rows=[]
    ordered_eval_df=eval_df.sort_values(["user_idx", TIME_COL, "interaction_id"]).reset_index(drop=True) #ordiniamo le righe da valutare
    for row in tqdm(ordered_eval_df.itertuples(index=False), total=len(ordered_eval_df), desc=f"Negative sampling {split_name}"): #scorriamo righe, con barra di scorrimento tqdm
        #itetuples trasforma ogni riga in un oggetto accessibile
        negatives=sample_neg_items( #campiona item negativi per la riga corrente
            user_idx=row.user_idx,
            positive_item_idx=row.item_idx, #item positivo corrente, creaiamo 100 negativi per ogni coppia utente-item positivo
            candidate_items=candidate_item_idx,
            seen_items_byuser=user_seen_items,
            n_neg=n_neg,
            rng=rng,
        )
        sample_rows.append({ #salva una riga di negative sampling
            "split":split_name,
            "user_idx":int(row.user_idx),
            "user_id":row.user_id,
            "positive_item_idx":int(row.item_idx),
            "positive_parent_asin":row.parent_asin,
            "negative_item_idxs":negatives,
            "num_negatives":int(len(negatives)),
        })
    return pd.DataFrame(sample_rows)

In [ ]:

validation_negative_samples_df=build_negative_samples(validation_eval_df, "validation", seed=RANDOM_STATE)#crea negativi validation
test_negatives_samples_df=build_negative_samples(test_eval_df, "test", seed=RANDOM_STATE+1) #negativi test ma seed diverso


print("validation_negative_samples_df:",validation_negative_samples_df)
print("test_negative_samples_df:",test_negatives_samples_df)
display(validation_negative_samples_df.head())
display(test_negatives_samples_df.head())


Negative sampling test: 100%|██████████| 131352/131352 [00:04<00:00, 28185.39it/s]


validation_negative_samples_df:              split  user_idx                       user_id  positive_item_idx  \
0       validation         0  AE22236AFRRSMQIKGG7TPTB75QEA               4617   
1       validation         1  AE222H3FGXWLHRFUMGMS2RR57NDQ               6050   
2       validation         2  AE223NW3QAU2KTAY5MAEU6VPBAUA               5807   
3       validation         3  AE224HM2QAW5TTSDL2QRDERA6KMA                343   
4       validation         4  AE2252DKW4XJIZP5QPFMQVJBVRTA               2444   
...            ...       ...                           ...                ...   
131371  validation    131393  AHZZXIAECG2VICHCSJEF5DRYKKSQ               1433   
131372  validation    131394  AHZZY6MRMU6PA6YHUWPHNRFO47QQ               2326   
131373  validation    131395  AHZZYR74YSTBTJMWGZFMU5B6HWZA               5387   
131374  validation    131396  AHZZZF7F5TA2BMROXTQTMUNV53RQ               6444   
131375  validation    131397  AHZZZY2XVWEUJUTYPGGL4WXH6CSA               2126

,split,user_idx,user_id,positive_item_idx,positive_parent_asin,negative_item_idxs,num_negatives
0,validation,0,AE22236AFRRSMQIKGG7TPTB75QEA,4617,B003ZDO2LM,"[259, 3414, 5268, 10203, 6251, 6543, 3981, 11457, 3205, 9702, 5687, 6513, 6868, 1934, 11272, 5301, 5897, 8066, 5849, 3315, 994, 780, 941...",100
1,validation,1,AE222H3FGXWLHRFUMGMS2RR57NDQ,6050,B0054JJ0QW,"[7710, 7535, 9359, 6955, 9631, 3850, 11767, 2880, 10843, 9619, 10135, 10058, 1190, 8283, 11116, 4856, 624, 1282, 10020, 1121, 1222, 1074...",100
2,validation,2,AE223NW3QAU2KTAY5MAEU6VPBAUA,5807,B004VRQNS6,"[7396, 842, 654, 6114, 7341, 9000, 11775, 6275, 8528, 1200, 3251, 189, 2318, 637, 8829, 9776, 1865, 5393, 10232, 1033, 7573, 12300, 8962...",100
3,validation,3,AE224HM2QAW5TTSDL2QRDERA6KMA,343,B0000DBJ8A,"[7445, 9637, 11790, 8567, 7608, 6313, 8444, 2044, 3338, 4990, 87, 6805, 7189, 1710, 3695, 5050, 5386, 8303, 11151, 7308, 298, 3115, 2303...",100
4,validation,4,AE2252DKW4XJIZP5QPFMQVJBVRTA,2444,B001L5SN7U,"[10098, 5608, 9372, 12247, 5178, 4924, 4000, 351, 5878, 3300, 6104, 1334, 14, 1952, 10369, 4984, 1346, 3484, 19, 1820, 5008, 2771, 4739,...",100


,split,user_idx,user_id,positive_item_idx,positive_parent_asin,negative_item_idxs,num_negatives
0,test,0,AE22236AFRRSMQIKGG7TPTB75QEA,6258,B005CG2AX2,"[1755, 11512, 9609, 3639, 2531, 12187, 11206, 10291, 7326, 11661, 11534, 5803, 1530, 10797, 484, 1813, 7682, 243, 5914, 1204, 1916, 1183...",100
1,test,1,AE222H3FGXWLHRFUMGMS2RR57NDQ,10679,B00CS43I4Q,"[11009, 8143, 10717, 3378, 4849, 12114, 12135, 5249, 9224, 6475, 12160, 8037, 513, 10464, 692, 6154, 442, 6947, 2167, 386, 10646, 11244,...",100
2,test,2,AE223NW3QAU2KTAY5MAEU6VPBAUA,4436,B003UE52ME,"[1116, 11006, 6577, 1340, 1046, 12143, 6196, 2954, 7190, 9536, 11628, 10305, 5432, 2177, 8942, 8506, 6333, 1822, 3066, 7956, 5188, 4127,...",100
3,test,3,AE224HM2QAW5TTSDL2QRDERA6KMA,469,B0002ILKNQ,"[7580, 2659, 2074, 2669, 10190, 5804, 7959, 6773, 1229, 764, 5136, 8933, 7019, 8107, 6821, 3954, 4480, 1080, 6183, 7438, 7522, 1914, 613...",100
4,test,4,AE2252DKW4XJIZP5QPFMQVJBVRTA,12183,B00IYA2QZA,"[4860, 1121, 3438, 11077, 7492, 9488, 5467, 4146, 3951, 11743, 4382, 765, 8031, 3808, 8934, 5943, 6028, 3448, 3848, 10231, 10561, 7451, ...",100


## 12 salvataggio degli output

In [ ]:
TRAIN_PATH=OUTPUT_DIR /"train.parquet"  #percorso train set
VALIDATION_PATH=OUTPUT_DIR/"validation.parquet" #percorso validation set originale
TEST_PATH=OUTPUT_DIR / "test.parquet"  #percorso test set originale
VALIDATION_EVAL_PATH=OUTPUT_DIR/"validation_eval.parquet" #percorso validation valutabile
TEST_EVAL_PATH=OUTPUT_DIR /"test_eval.parquet" #percorso test valutabile
ITEMS_OUT_PATH=OUTPUT_DIR /"items.parquet" #percorso prodotti finali
QA_OUT_PATH=OUTPUT_DIR / "qa.parquet"  #percorso QA finale
USER_MAPPING_PATH=OUTPUT_DIR /"user_mapping.parquet"  #percorso mapping utenti
ITEM_MAPPING_PATH=OUTPUT_DIR /"item_mapping.parquet"  #percorso mapping prodotti
VALIDATION_NEGATIVE_PATH=OUTPUT_DIR / "negative_samples_validation.parquet"  #percorso negativi validation
TEST_NEGATIVE_PATH=OUTPUT_DIR / "negative_samples_test.parquet" #percorso negativi test
SPLIT_SUMMARY_PATH=OUTPUT_DIR /"split_summary.json" #percorso riepilogo finale
FILTER_HISTORY_PATH=OUTPUT_DIR /"positive_filter_history.csv" #percorso storia del filtro

In [ ]:
train_df.to_parquet(TRAIN_PATH, index=False)#salva train set
validation_df.to_parquet(VALIDATION_PATH, index=False) #salva validation set originale
test_df.to_parquet(TEST_PATH, index=False) #salva test set originale
validation_eval_df.to_parquet(VALIDATION_EVAL_PATH, index=False)  #salva validation valutabile
test_eval_df.to_parquet(TEST_EVAL_PATH, index=False)  #salva test valutabile
final_items_df.to_parquet(ITEMS_OUT_PATH, index=False)  #salva prodotti finali
final_qa_df.to_parquet(QA_OUT_PATH, index=False) #salva QA finale
user_mapping_df.to_parquet(USER_MAPPING_PATH, index=False)#salva mapping utenti
item_mapping_df.to_parquet(ITEM_MAPPING_PATH, index=False)#salva mapping prodotti

validation_negative_samples_df.to_parquet(VALIDATION_NEGATIVE_PATH,index=False)
test_negatives_samples_df.to_parquet(TEST_NEGATIVE_PATH, index=False)

filter_history_df.to_csv(FILTER_HISTORY_PATH, index=False)#salva storia del filtro positivo


In [69]:
def density_percentage(num_rows,num_users,num_items):
    possible_pairs=num_users*num_items #calcola numero coppie utente-prodotto possibili
    if possible_pairs==0:
        return None 
    return float(num_rows/possible_pairs*100)

In [ ]:
split_summary={  #crea riepilogo finale da salvare
    "category":CATEGORY,
    "source_config": SELECTED_CONFIG,  #salva configurazione di partenza del Notebook 02
    "positive_rating_threshold":int(POSITIVE_RATING_THRESHOLDS),  #salva soglia rating positivo
    "min_user_interactions_after":int(MIN_USER_INTERACTIONS), #salva soglia utente dopo target
    "min_item_interactions_after": int(MIN_ITEM_INTERACTIONS),#salva soglia prodotto dopo target
    "n_negatives": int(N_NEG), #salva numero negativi
    "random_state": int(RANDOM_STATE),  #salva seed
    "temporal_column": TIME_COL,  #salva la colonna temporale usata per lo split
    "original_candidate_interactions": int(len(interactions_df)),#salva interazioni candidate iniziali
    "positive_interactions_before_refilter": int(len(positive_interactions_df)),  #salva positive prima del rifiltro
    "final_interactions": int(len(indexed_interactions_df)),  #salva interazioni finali
    "final_users":int(indexed_interactions_df["user_idx"].nunique()),
    "final_items":int(indexed_interactions_df["item_idx"].nunique()),
    "final_density_pct": density_percentage(len(indexed_interactions_df), indexed_interactions_df["user_idx"].nunique(), indexed_interactions_df["item_idx"].nunique()),
    "train_rows":int(len(train_df)),
    "validation_rows": int(len(validation_df)),
    "test_rows":int(len(test_df)),
    "validation_eval_rows": int(len(validation_eval_df)),#salva righe validation valutabili
    "test_eval_rows": int(len(test_eval_df)),#salva righe test valutabili
    "validation_cold_items": int(len(validation_cold_items)),  #salva item cold-start validation
    "test_cold_items": int(len(test_cold_items)),  # salva item cold-start test
    "train_path": str(TRAIN_PATH),  # salva percorso train
    "validation_eval_path": str(VALIDATION_EVAL_PATH),  #salva percorso validation eval
    "test_eval_path": str(TEST_EVAL_PATH),  #salva percorso test eval
    "negative_samples_validation_path": str(VALIDATION_NEGATIVE_PATH),  #salva percorso negativi validation
    "negative_samples_test_path": str(TEST_NEGATIVE_PATH),  #salva percorso negativi test
}  # chiude dizionario riepilogo.

SPLIT_SUMMARY_PATH.write_text(json.dumps(split_summary,indent=2,ensure_ascii=False),encoding="utf-8")  #salva riepilogo JSON
display(pd.DataFrame.from_dict(split_summary,orient="index",columns=["value"]))

,value
category,Electronics
source_config,user3_item8
positive_rating_threshold,4
min_user_interactions_after,3
min_item_interactions_after,8
n_negatives,100
random_state,42
temporal_column,datetime
original_candidate_interactions,844060
positive_interactions_before_refilter,671903
